# Blocs in the wave — Martin's exploration

**02805 Social Graphs and Interactions, project exploration (Martin). A prototype, not the paper.**

**Question.** Is the House Republican party one bloc or several: do the blocs found in the cosponsorship
network also speak differently, and did the 2026 wave hit them equally?

This notebook computes every number and then writes **`martin_blocs.html`**, an interactive page that tells
the same story as you scroll. Open that file in a browser after running this.

It reads only `data/processed/` and reuses the group's definitions from `walk_or_talk.ipynb` (bill types,
stop lists, study population, overperformance) so the two notebooks stay comparable.

**Honesty note.** Before this notebook was written, a quick probe on the same data had already shown that
blocs exist and that overperformance differed between them (unadjusted p = 0.002). The definitions below were
fixed after that probe, so step 5 is exploratory, not a clean test. The state check in step 5 was added on
11 November after the first run of this notebook showed that the blocs which ran ahead are built around
state delegations.

## 0. Every modelling choice in one place

In [1]:
import json, re, math
import numpy as np, pandas as pd, networkx as nx
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import normalized_mutual_info_score as nmi

DATA, SEED = "data/processed", 2805
rng = np.random.default_rng(SEED)

PARTY            = "R"              # the party we look inside (Democrats are run once as a contrast)
BILL_TYPES       = ("hr", "hjres")  # as in walk_or_talk: simple/concurrent resolutions are mostly commemorative
ALPHA            = 0.05             # disparity filter: keep a tie if it is this surprising for either member
N_LOUVAIN        = 100              # Louvain runs with different seeds; we keep the most typical partition
N_REWIRE         = 200              # degree-preserving rewirings for the structure null
MIN_BLOC         = 8                # blocs smaller than this are reported but not tested or coloured
MIN_TURN_WORDS   = 50               # text: as in walk_or_talk
MIN_MEMBER_WORDS = 3000             # text: as in walk_or_talk
MIN_WORD_MEMBERS = 20               # text: as in walk_or_talk
SMOOTH           = 0.1              # text: as in walk_or_talk
N_SHUFFLES       = 5000             # election null
N_TEXT_SHUFFLES  = 2000             # text null
LEAN_GROUPS      = 5                # election null shuffles within fifths of district lean

## 1. Data and the within-party network

In [2]:
members    = pd.read_csv(f"{DATA}/members.csv").set_index("bioguide")
bills      = pd.read_parquet(f"{DATA}/bills.parquet")
cosponsors = pd.read_parquet(f"{DATA}/cosponsors.parquet")
speeches   = pd.read_parquet(f"{DATA}/speeches.parquet")
districts  = pd.read_csv(f"{DATA}/districts.csv")
incumbents = pd.read_csv(f"{DATA}/incumbents.csv")
party = members.party
print("members by party:", party.value_counts().to_dict())

b = bills[bills.type.isin(BILL_TYPES)]
links = cosponsors[~cosponsors.withdrawn].merge(b[["bill_id", "sponsor"]], on="bill_id")
links = links[links.cosponsor.isin(members.index) & links.sponsor.isin(members.index) & (links.cosponsor != links.sponsor)]
links = links.assign(cross=party.reindex(links.cosponsor).values != party.reindex(links.sponsor).values)
walk = links.groupby("cosponsor").cross.mean().rename("walk")        # the group's earlier score, kept for comparison
print(f"cosponsorships used: {len(links):,} on {links.bill_id.nunique():,} bills; {links.cross.mean():.0%} cross the aisle")

def party_graph(p):
    """Undirected graph of one party. Tie weight = number of times either member cosponsored the other's bill."""
    same = links[(party.reindex(links.cosponsor).values == p) & (party.reindex(links.sponsor).values == p)]
    pair = pd.DataFrame({"a": np.minimum(same.cosponsor.values, same.sponsor.values),
                         "b": np.maximum(same.cosponsor.values, same.sponsor.values)}).value_counts()
    G = nx.Graph(); G.add_nodes_from(party.index[party == p])
    G.add_weighted_edges_from((a, c, int(w)) for (a, c), w in pair.items())
    return G

G = party_graph(PARTY)
print(f"{PARTY} network: {G.number_of_nodes()} members, {G.number_of_edges():,} ties, density {nx.density(G):.2f}")

members by party: {'R': 226, 'D': 224, 'I': 1}
cosponsorships used: 126,687 on 8,714 bills; 21% cross the aisle
R network: 226 members, 14,800 ties, density 0.58


**Choice: one party only.** Ties to the other party are dropped here, because the party split dominates
everything else (Louvain on the whole House finds the two parties). *Alternative:* keep both parties and
raise the Louvain resolution; blocs would then mix "who is moderate" with "who is in which party".

**Choice: undirected, counted ties.** A tie's weight is how often two members signed each other's bills.
*Alternative:* keep direction (cosponsor → sponsor), which separates leaders from followers but needs a
directed community method.

## 2. Backbone and blocs

The raw network is dense (more than half of all pairs are tied), so communities are hard to see. The
**disparity filter** (week 4) keeps a tie only if it carries a surprisingly large share of a member's total
weight, given how many ties that member has. A tie is kept if it passes for **either** member.

In [3]:
def backbone(G, alpha):
    H = nx.Graph(); H.add_nodes_from(G)
    for u in G:
        k, s = G.degree(u), G.degree(u, weight="weight")
        if k < 2: continue                      # a member with one tie cannot be tested
        for v in G[u]:
            if (1 - G[u][v]["weight"] / s) ** (k - 1) < alpha:
                H.add_edge(u, v, weight=G[u][v]["weight"])
    H.remove_nodes_from([n for n in list(H) if H.degree(n) == 0])
    return H

def louvain_runs(H, n, weight=None):
    nodes = list(H)
    parts = [nx.community.louvain_communities(H, weight=weight, seed=s) for s in range(n)]
    labs = np.array([[next(k for k, c in enumerate(p) if v in c) for v in nodes] for p in parts])
    Q = np.array([nx.community.modularity(H, p, weight=weight) for p in parts])
    agree = np.array([[nmi(labs[i], labs[j]) for j in range(n)] for i in range(n)])
    typical = int(np.argmax(agree.sum(1)))      # the run that agrees most with all other runs
    stability = float(agree[np.triu_indices(n, 1)].mean())
    return dict(nodes=nodes, labels=labs[typical], Q=float(Q[typical]), Q_all=Q, stability=stability,
                n_blocs=[len(p) for p in parts])

def rewired_null(H, n):
    """Louvain modularity of networks with the same degrees but ties placed at random."""
    out = []
    for s in range(n):
        R = H.copy(); m = R.number_of_edges()
        nx.double_edge_swap(R, nswap=10 * m, max_tries=1000 * m, seed=s)
        out.append(nx.community.modularity(R, nx.community.louvain_communities(R, weight=None, seed=s), weight=None))
    return np.array(out)

H = backbone(G, ALPHA)
run = louvain_runs(H, N_LOUVAIN)
null_Q = rewired_null(H, N_REWIRE)
print(f"backbone: {H.number_of_nodes()} members, {H.number_of_edges()} ties ({G.number_of_nodes() - H.number_of_nodes()} members keep no tie)")
print(f"Louvain modularity {run['Q']:.3f}; rewired null {null_Q.mean():.3f} ± {null_Q.std():.3f} (max {null_Q.max():.3f})")
print(f"number of blocs over {N_LOUVAIN} runs: {pd.Series(run['n_blocs']).value_counts().sort_index().to_dict()}")
print(f"agreement between runs (mean NMI): {run['stability']:.2f}")

raw = pd.Series(run["labels"], index=run["nodes"])
order = raw.value_counts().index                         # largest bloc first
letter = {k: chr(65 + i) for i, k in enumerate(order)}
bloc = raw.map(letter).rename("bloc")
sizes = bloc.value_counts()
big = [k for k in sizes.index if sizes[k] >= MIN_BLOC][:8]
print("bloc sizes:", sizes.to_dict()); print("tested blocs:", big)
assert bloc.index.is_unique and set(bloc.index) == set(H)

backbone: 215 members, 659 ties (11 members keep no tie)
Louvain modularity 0.532; rewired null 0.358 ± 0.006 (max 0.372)
number of blocs over 100 runs: {8: 2, 9: 20, 10: 43, 11: 31, 12: 4}
agreement between runs (mean NMI): 0.76
bloc sizes: {'A': 43, 'B': 36, 'C': 29, 'D': 26, 'E': 23, 'F': 18, 'G': 17, 'H': 10, 'I': 7, 'J': 3, 'K': 3}
tested blocs: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']


**What modularity captures:** more ties inside groups than the degrees alone would give. **What it misses:**
it says the grouping is there, not that it is the right one. The rewired null is the check: the same
members with the same number of ties, placed at random.

**Choice: which partition.** Louvain gives a different answer per seed. We run it 100 times and keep the run
that agrees most with the others. *Alternative:* the run with the highest modularity, which is less typical.

### Does the answer survive other settings?

In [4]:
def compare(label, other_nodes, other_labels, Q, n):
    o = pd.Series(other_labels, index=other_nodes); common = bloc.index.intersection(o.index)
    return dict(setting=label, members=len(o), blocs=n, modularity=round(Q, 3), nmi_with_main=round(float(nmi(bloc[common], o[common])), 2))

rows = [dict(setting=f"main: backbone α = {ALPHA}", members=len(bloc), blocs=bloc.nunique(), modularity=round(run["Q"], 3), nmi_with_main=1.0)]
for a in (0.01, 0.10):
    Ha = backbone(G, a); ra = louvain_runs(Ha, 30)
    rows.append(compare(f"backbone α = {a}", ra["nodes"], ra["labels"], ra["Q"], len(set(ra["labels"]))))
rw = louvain_runs(G, 30, weight="weight")
rows.append(compare("no backbone, weighted Louvain", rw["nodes"], rw["labels"], rw["Q"], len(set(rw["labels"]))))
sensitivity = pd.DataFrame(rows); print(sensitivity.to_string(index=False))

# contrast: the same recipe inside the Democratic party
GD = party_graph("D"); HD = backbone(GD, ALPHA); rD = louvain_runs(HD, 30); nD = rewired_null(HD, 50)
print(f"\nDemocrats: modularity {rD['Q']:.3f}; rewired null {nD.mean():.3f} ± {nD.std():.3f}; agreement between runs {rD['stability']:.2f}")

                      setting  members  blocs  modularity  nmi_with_main
      main: backbone α = 0.05      215     11       0.532           1.00
            backbone α = 0.01      143     19       0.747           0.68
             backbone α = 0.1      222      8       0.390           0.50
no backbone, weighted Louvain      226      7       0.154           0.44



Democrats: modularity 0.358; rewired null 0.290 ± 0.005; agreement between runs 0.56


## 3. What are the blocs? Ideology and geography

Two boring explanations: a bloc is just a slice of the left–right scale, or just a state delegation.
- **Ideology:** share of the variation in the roll-call score (NOMINATE, first dimension) that bloc explains (η²).
- **Geography:** agreement between bloc and state (NMI), against the same blocs with labels shuffled.

In [5]:
def eta2(v, g):
    v, g = np.asarray(v, float), np.asarray(g)
    tot = ((v - v.mean()) ** 2).sum()
    return float(sum(((v[g == k].mean() - v.mean()) ** 2) * (g == k).sum() for k in np.unique(g)) / tot)

T = members.loc[bloc.index].join(bloc).join(walk)
Tb = T[T.bloc.isin(big)]
ok = Tb.nominate_dim1.notna()
ideo_eta = eta2(Tb.nominate_dim1[ok], Tb.bloc[ok])
ideo_null = np.array([eta2(Tb.nominate_dim1[ok], rng.permutation(Tb.bloc[ok].to_numpy(dtype=object))) for _ in range(1000)])
state_nmi = float(nmi(Tb.bloc, Tb.state))
state_null = np.array([nmi(rng.permutation(Tb.bloc.to_numpy(dtype=object)), Tb.state) for _ in range(1000)])
print(f"ideology explained by bloc: η² = {ideo_eta:.2f} (shuffled labels: {ideo_null.mean():.2f}, 95% below {np.quantile(ideo_null, .95):.2f})")
print(f"bloc vs state: NMI = {state_nmi:.2f} (shuffled labels: {state_null.mean():.2f} ± {state_null.std():.2f})")
print(Tb.groupby("bloc").agg(n=("name", "size"), ideology=("nominate_dim1", "mean"), walk=("walk", "mean"),
                              top_state=("state", lambda s: f"{s.value_counts().index[0]} {s.value_counts().iloc[0]}")).round(2).to_string())

ideology explained by bloc: η² = 0.35 (shuffled labels: 0.03, 95% below 0.07)
bloc vs state: NMI = 0.41 (shuffled labels: 0.24 ± 0.01)
       n  ideology  walk top_state
bloc                              
A     43      0.67  0.06      TX 8
B     36      0.38  0.27      NY 6
C     29      0.49  0.13      UT 4
D     26      0.58  0.10      NC 8
E     23      0.53  0.10     TX 10
F     18      0.53  0.10      WI 6
G     17      0.53  0.11     FL 11
H     10      0.46  0.15      MI 5


## 4. Do the blocs speak differently?

Text is noisy for one member (the group's split-half check gave ρ = 0.41) but we now compare **groups**.

- Same cleaning and vocabulary as `walk_or_talk.ipynb`.
- Each member becomes a TF-IDF vector (week 6). Similarity between two members is the cosine of their vectors.
- **Statistic:** mean similarity of two members in the same bloc minus mean similarity of two members in different blocs.
- **Null:** shuffle the bloc labels over the same members.
- **Guard against geography:** the same statistic using only pairs from **different states**, because
  colleagues from one state share place names and local topics.

**Captures:** shared vocabulary on the House floor. **Misses:** stance (two members can use the same word
on opposite sides), and everything said outside the floor.

In [6]:
PROCEDURAL = set("""speaker madam mister chair chairman chairwoman gentleman gentlewoman gentlelady yield yields yielding yielded
    time balance reserve reserves consume unanimous consent rise rises minute minutes hour bill bills resolution amendment
    amendments committee subcommittee house senate congress congressional colleague colleagues thank thanks urge support
    vote votes voting legislation legislative motion suspend rules pass passage revise extend remarks extraneous material
    days day members member act title section pursuant clerk rule order ordered objection question previous agreed ayes nays
    yeas recorded demand postponed proceedings further request recognize recognized ranking distinguished friend
    representative representatives rep sen senator today want would like also one may shall mrs dr hon res con""".split())
MONTHS = set("january february march april may june july august september october november december monday tuesday wednesday thursday friday saturday sunday".split())
name_words = {w for col in ("first", "last", "name") for s in members[col].dropna() for w in re.findall(r"[a-z]{3,}", s.lower())}
STATE_NAMES = """alabama alaska arizona arkansas california colorado connecticut delaware florida georgia hawaii idaho illinois indiana
    iowa kansas kentucky louisiana maine maryland massachusetts michigan minnesota mississippi missouri montana nebraska nevada
    hampshire jersey mexico york carolina dakota ohio oklahoma oregon pennsylvania rhode island tennessee texas utah vermont
    virginia washington wisconsin wyoming north south west new""".split()
KEEP = {"tax", "price", "rich", "young", "green", "white", "black", "brown", "law", "bacon", "energy", "freedom", "justice", "border", "trade", "bill"} - PROCEDURAL
STOP = (set(ENGLISH_STOP_WORDS) | PROCEDURAL | MONTHS | set(STATE_NAMES) | name_words) - KEEP      # copied from walk_or_talk.ipynb

sp = speeches[speeches.n_words >= MIN_TURN_WORDS]
docs = sp.groupby("bioguide").text.agg(" ".join)
n_words = sp.groupby("bioguide").n_words.sum()
cv = CountVectorizer(token_pattern=r"(?u)\b[a-z]{3,}\b", lowercase=True, stop_words=list(STOP), min_df=MIN_WORD_MEMBERS)
X = cv.fit_transform(docs).toarray().astype(float)
vocab = np.array(cv.get_feature_names_out())
rates = 1e4 * X / X.sum(1, keepdims=True)
row = {m: i for i, m in enumerate(docs.index)}

ids = [m for m in Tb.index if m in row and n_words[m] >= MIN_MEMBER_WORDS]          # Republicans in a tested bloc with enough text
lab = Tb.bloc[ids].to_numpy(dtype=object); st = Tb.state[ids].to_numpy(dtype=object)
idf = np.log(len(docs) / (X > 0).sum(0))
V = rates[[row[m] for m in ids]] * idf
V /= np.linalg.norm(V, axis=1, keepdims=True)
S = V @ V.T
iu = np.triu_indices(len(ids), 1)
sim, other_state = S[iu], (st[:, None] != st[None, :])[iu]

def gap(labels, mask=None):
    same = (labels[:, None] == labels[None, :])[iu]
    if mask is not None: same, s = same[mask], sim[mask]
    else: s = sim
    return float(s[same].mean() - s[~same].mean())

text = {}
for key, mask in [("all", None), ("other_state", other_state)]:
    obs = gap(lab, mask); null = np.array([gap(rng.permutation(lab), mask) for _ in range(N_TEXT_SHUFFLES)])
    text[key] = dict(obs=obs, null=null, p=float((null >= obs).mean()))
    print(f"{key:<12} same-bloc minus different-bloc similarity = {obs:+.4f}; null {null.mean():+.4f} ± {null.std():.4f}; p = {text[key]['p']:.4f}")
print(f"members compared: {len(ids)} | mean similarity of any two: {sim.mean():.3f}")

# the words that set each bloc apart from the rest of the party (the group's week-6 measure, every member counts once)
allR = [m for m in members.index[party == PARTY] if m in row and n_words[m] >= MIN_MEMBER_WORDS]
words = {}
for k in big:
    mine = set(Tb.index[Tb.bloc == k])
    inn = [row[m] for m in ids if m in mine]; out = [row[m] for m in allR if m not in mine]
    score = np.log2((rates[inn].mean(0) + SMOOTH) / (rates[out].mean(0) + SMOOTH))
    used_by = (X[inn] > 0).sum(0)
    keep = used_by >= max(3, math.ceil(len(inn) / 3))          # a word must be used by a third of the bloc, so one member cannot own it
    top = np.argsort(-np.where(keep, score, -np.inf))[:12]
    words[k] = [(vocab[j], round(float(score[j]), 2), int(used_by[j])) for j in top]
    print(k, len(inn), "members:", ", ".join(vocab[j] for j in top))

all          same-bloc minus different-bloc similarity = +0.0121; null -0.0001 ± 0.0018; p = 0.0000


other_state  same-bloc minus different-bloc similarity = +0.0101; null -0.0000 ± 0.0019; p = 0.0000
members compared: 188 | mean similarity of any two: 0.106
A 39 members: lord, constitution, prayer, bible, radical, oath, sex, alien, jesus, constitutional, gender, powers
B 34 members: russian, ukraine, civilians, russia, sanctions, eagle, jewish, bay, democracy, regime, terror, beach
C 26 members: irs, deduction, unemployment, root, refunds, double, doubled, wages, tax, corn, watched, educators
D 24 members: grid, reliability, nomination, prescription, exports, worth, pipelines, hurricane, emissions, pharmacy, pipeline, operators
E 23 members: dallas, regulators, formation, banks, entrepreneurs, jones, codifying, multi, regulated, capital, texans, financial
F 17 members: hockey, device, fish, amended, disability, restrict, honestly, olympic, accessing, olympics, developers, claims
G 16 members: sunshine, waterways, coast, haven, maritime, checks, beach, garden, port, surface, girl, app

## 5. Did the wave hit the blocs equally?

Population and outcome are the group's (`walk_or_talk.ipynb`, section 3): Republican incumbents who were the
only incumbent, against exactly one Democrat, in a called race. **Overperformance** = two-party margin minus
a straight-line prediction from district lean.

- **Statistic:** share of the variation in overperformance that bloc explains (η²).
- **Null:** shuffle overperformance among members whose districts have similar lean (five groups), 5,000 times.
- **Second look:** the same without the redrawn states, where the lean figure is unverified.
- **State check:** several blocs are built around state delegations, so a bloc that ran ahead may only be a
  state that swung less. The same statistic is tested against a second null: shuffle overperformance among
  members **of the same state**. This keeps every state's results in that state and asks whether bloc adds
  anything inside states. Only members who share a state with a member of another bloc can move, so this
  test has less power. *Alternative:* subtract each state's mean first; that needs states with several
  members and gives the same reading.

**Captures:** whether some blocs ran ahead of their districts as a group. **Misses:** why. Challenger
quality, money and local issues are not in the data, and nothing here is causal.

In [7]:
inc = incumbents.merge(districts, on=["state", "district"], how="left").set_index("bioguide")
inc["party"] = party.reindex(inc.index)
n_inc = inc.groupby(["state", "district"]).size()
inc["n_incumbents"] = [n_inc[(s, d)] for s, d in zip(inc.state, inc.district)]
rep = inc[inc.party == PARTY].copy()
out = rep.fate.isin(["did not run", "lost primary"]) | (rep.n_incumbents > 1) | (rep.n_rep != 1) | (rep.n_dem != 1) | rep.winner.isna()
pop = rep[~out].copy()
pop["margin"] = 100 * (pop.rep_votes - pop.dem_votes) / (pop.rep_votes + pop.dem_votes)
slope, intercept = np.polyfit(pop.lean, pop.margin, 1)
pop["over"] = pop.margin - (intercept + slope * pop.lean)
tx1 = districts[(districts.state == "TX") & (districts.district == 1)].iloc[0]
assert (tx1.rep_votes, tx1.dem_votes) == (181987, 81598), "TX-1 spot check failed: the results file has changed"
print(f"study population: {len(pop)} (walk_or_talk reported 162) | lost: {int((pop.winner != 'gop').sum())}")
print(f"margin = {intercept:+.1f} {slope:+.2f} × lean, R² = {np.corrcoef(pop.lean, pop.margin)[0, 1] ** 2:.2f}")

E = pop.join(bloc, how="inner"); E = E[E.bloc.isin(big)]

def election_test(d, within="lean"):
    """Shuffle overperformance among members in districts of similar lean, or among members of the same state."""
    g = pd.qcut(d.lean.rank(method="first"), LEAN_GROUPS, labels=False).values if within == "lean" else pd.factorize(d.state)[0]
    idx = [np.flatnonzero(g == k) for k in np.unique(g)]
    y, b_ = d.over.values, d.bloc.to_numpy(dtype=object)
    movable = sum(len(ix) for ix in idx if len(set(b_[ix])) > 1)      # members whose shuffle group holds more than one bloc
    obs = eta2(y, b_); null = np.empty(N_SHUFFLES)
    for s in range(N_SHUFFLES):
        ys = y.copy()
        for ix in idx: ys[ix] = y[rng.permutation(ix)]
        null[s] = eta2(ys, b_)
    return dict(n=len(d), obs=obs, null=null, p=float((null >= obs).mean()), movable=int(movable))

elec = {"all": election_test(E), "no_redrawn": election_test(E[~E.redrawn_state]),
        "state": election_test(E, "state"), "state_no_redrawn": election_test(E[~E.redrawn_state], "state")}
for k, r in elec.items():
    print(f"{k:<17} n = {r['n']} ({r['movable']} can move): bloc explains η² = {r['obs']:.3f} of overperformance; null {r['null'].mean():.3f}, 95% below {np.quantile(r['null'], .95):.3f}; p = {r['p']:.4f}")
state_eta = eta2(E.over, E.state.to_numpy(dtype=object))
state_eta_null = np.array([eta2(rng.permutation(E.over.values), E.state.to_numpy(dtype=object)) for _ in range(1000)])
print(f"state alone explains η² = {state_eta:.3f} of overperformance (shuffled: {state_eta_null.mean():.3f}, 95% below {np.quantile(state_eta_null, .95):.3f}); {E.state.nunique()} states")

fate = rep.fate.reindex(bloc.index).fillna("left mid-term")
summary = pd.DataFrame({"n": sizes, "in_study": E.groupby("bloc").size(), "mean_over": E.groupby("bloc").over.mean(),
                        "lost_general": (fate == "lost general").groupby(bloc).sum(),
                        "left_before_vote": fate.isin(["did not run", "lost primary"]).groupby(bloc).sum(),
                        "mean_lean": rep.lean.reindex(bloc.index).groupby(bloc).mean()}).round(2)
print(summary.to_string())

study population: 162 (walk_or_talk reported 162) | lost: 30
margin = -11.2 +1.87 × lean, R² = 0.83


all               n = 149 (149 can move): bloc explains η² = 0.168 of overperformance; null 0.046, 95% below 0.089; p = 0.0000
no_redrawn        n = 93 (93 can move): bloc explains η² = 0.182 of overperformance; null 0.078, 95% below 0.154; p = 0.0156
state             n = 149 (122 can move): bloc explains η² = 0.168 of overperformance; null 0.165, 95% below 0.202; p = 0.4198
state_no_redrawn  n = 93 (70 can move): bloc explains η² = 0.182 of overperformance; null 0.190, 95% below 0.234; p = 0.6038


state alone explains η² = 0.663 of overperformance (shuffled: 0.235, 95% below 0.331); 36 states
       n  in_study  mean_over  lost_general  left_before_vote  mean_lean
bloc                                                                    
A     43      26.0      -1.75             5                11      12.15
B     36      30.0       0.38            13                 4       6.56
C     29      18.0      -1.26             2                10      13.93
D     26      22.0      -2.70             6                 2      12.42
E     23      18.0      -2.03             2                 4      12.82
F     18      13.0       4.50             3                 3      11.06
G     17      14.0       2.07             0                 3      11.65
H     10       8.0       5.11             1                 2      10.90
I      7       NaN        NaN             1                 1       5.20
J      3       NaN        NaN             0                 0      23.67
K      3       NaN        N

## 6. Write the interactive page

Everything the page shows comes from the variables above.

**Choice: the drawing.** A plain spring layout of the backbone is one dense ball. Here ties inside a bloc
pull six times harder than ties between blocs, and far-out members are pulled inward. This makes the blocs
visible and also makes them look **more separate than they are**. Positions carry no measurement, and no
number in this notebook depends on them.

In [8]:
for u, v in H.edges: H[u][v]["pull"] = 3.0 if bloc[u] == bloc[v] else 0.5      # drawing only: ties inside a bloc pull harder
pos = nx.spring_layout(H, seed=SEED, iterations=600, k=3.2 / math.sqrt(len(H)), weight="pull")
xy = np.array([pos[n] for n in H])
c = xy - np.median(xy, 0); r = np.linalg.norm(c, axis=1) / np.linalg.norm(c, axis=1).max()
c = c * (np.sqrt(r) / np.maximum(r, 1e-9))[:, None]                               # pull the far-out chains inward so the middle is readable
xy = (c - c.min(0)) / (c.max(0) - c.min(0))
nodes_ = list(H); at = {n: i for i, n in enumerate(nodes_)}
def num(v, d=2): return None if v is None or (isinstance(v, float) and math.isnan(v)) else round(float(v), d)

nodes = []
for n in nodes_:
    m = members.loc[n]; r = rep.loc[n] if n in rep.index else None
    nodes.append(dict(id=n, name=m["name"], state=m.state, district=int(m.district), bloc=bloc[n], big=bool(bloc[n] in big),
        x=num(xy[at[n], 0], 4), y=num(xy[at[n], 1], 4), ideology=num(m.nominate_dim1, 3), walk=num(walk.get(n, np.nan), 3),
        words=int(n_words.get(n, 0)), degree=H.degree(n), fate=fate[n],
        lean=num(r.lean, 0) if r is not None else None, redrawn=bool(r.redrawn_state) if r is not None else None,
        margin=num(pop.margin.get(n, np.nan), 1), over=num(pop.over.get(n, np.nan), 1)))

blocs = []
for k in sizes.index:
    t = T[T.bloc == k]; s = t.state.value_counts()
    blocs.append(dict(id=k, n=int(sizes[k]), big=bool(k in big), ideology=num(t.nominate_dim1.mean(), 3), walk=num(t.walk.mean(), 3),
        lean=num(summary.mean_lean[k], 1), states=[[a, int(c)] for a, c in s.head(4).items()],
        top_state_share=num(s.iloc[0] / len(t), 2), words=words.get(k, []), fates={a: int(c) for a, c in fate[bloc == k].value_counts().items()},
        in_study=int(summary.in_study.fillna(0)[k]), over=num(summary.mean_over[k], 1)))

payload = dict(
    meta=dict(party=PARTY, alpha=ALPHA, min_bloc=MIN_BLOC, n_party=G.number_of_nodes(), n_ties=G.number_of_edges(),
              density=num(nx.density(G)), n_backbone=H.number_of_nodes(), m_backbone=H.number_of_edges(),
              n_links=len(links), cross=num(links.cross.mean()), n_louvain=N_LOUVAIN, n_rewire=N_REWIRE,
              results_date="2026-11-11", pct_in_min=num(pop.pct_in.min(), 0)),
    nodes=nodes, edges=[[at[u], at[v], int(d["weight"])] for u, v, d in H.edges(data=True)], blocs=blocs, big=big,
    structure=dict(Q=num(run["Q"], 3), null=[num(v, 3) for v in null_Q], stability=num(run["stability"]),
                   n_blocs={int(a): int(c) for a, c in pd.Series(run["n_blocs"]).value_counts().sort_index().items()},
                   dem=dict(Q=num(rD["Q"], 3), null_mean=num(nD.mean(), 3), null_sd=num(nD.std(), 3), stability=num(rD["stability"]))),
    sensitivity=json.loads(sensitivity.to_json(orient="records")),
    what=dict(ideo_eta=num(ideo_eta), ideo_null95=num(np.quantile(ideo_null, .95)), state_nmi=num(state_nmi),
              state_null=num(state_null.mean()), state_null_sd=num(state_null.std(), 3)),
    text={k: dict(obs=num(v["obs"], 4), null=[num(x, 4) for x in v["null"]], p=num(v["p"], 4)) for k, v in text.items()} | dict(n=len(ids), mean_sim=num(sim.mean(), 3)),
    election={k: dict(n=v["n"], obs=num(v["obs"], 3), null=[num(x, 3) for x in v["null"]], p=num(v["p"], 4), movable=v["movable"]) for k, v in elec.items()}
             | dict(state_eta=num(state_eta, 3), state_eta_null95=num(np.quantile(state_eta_null, .95), 3), n_states=int(E.state.nunique()), slope=num(slope, 3), intercept=num(intercept, 2), n_pop=len(pop), r2=num(np.corrcoef(pop.lean, pop.margin)[0, 1] ** 2)),
)
html = open("martin_blocs_template.html", encoding="utf-8").read().replace("__DATA__", json.dumps(payload, ensure_ascii=False))
open("martin_blocs.html", "w", encoding="utf-8").write(html)
print(f"wrote martin_blocs.html ({len(html) / 1e3:.0f} kB): {len(nodes)} members, {len(payload['edges'])} ties, {len(blocs)} blocs")

wrote martin_blocs.html (286 kB): 215 members, 659 ties, 11 blocs
